# Reproduce AI Competitiveness Baseline

## Phase Bridge (Phase 1: Baseline Rebuild and Validation)
This notebook executes the baseline reconstruction stage that all downstream phases depend on. It rebuilds country and KPI baseline scores, validates key consistency targets, and exports canonical baseline artifacts.

### Workflow placement
- Depends on: packaged workbook and Canada gap CSV
- Produces: baseline_country_scores.csv, baseline_kpi_scores.csv, validation_report.md
- Used by later phases: impact mapping (Phase 3), simulation (Phase 4), attribution (Phase 5), dashboard/report layers (Phase 6)

## Why this phase is necessary
Policy simulation and attribution are only meaningful if baseline scores, KPI weights, and rank ordering are reproduced deterministically first.

## Main inputs
- data/raw/AI_Competitiveness_Rankings_and_Weights (1) - Copy.xlsx
  - required sheets include KPI Scores and Weights and Final Rankings
- data/raw/canada_combined_kpi_weights_gaps - Copy.csv

## Core formulas and logic
- weighted_kpi_score = kpi_score * kpi_weight
- subpillar_score = sum(weighted_kpi_score within subpillar)
- weighted_subpillar_score = subpillar_score * subpillar_weight
- composite_score = sum(weighted_subpillar_score across subpillars)
- ranks are recomputed from descending composite_score

## Assumptions
- Workbook values are treated as scoring-ready in this phase.
- Country universe is fixed to 13 countries.
- Required workbook schema/labels must match expected structure.

In [ ]:
from pathlib import Path

from notebook_setup import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


In [ ]:
from src.scoring.baseline_reproduction import reproduce_baseline

WORKBOOK_PATH = PROJECT_ROOT / "data" / "raw" / "AI_Competitiveness_Rankings_and_Weights (1) - Copy.xlsx"
CANADA_GAP_PATH = PROJECT_ROOT / "data" / "raw" / "canada_combined_kpi_weights_gaps - Copy.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"


## Method Overview and Validation/QA Coverage

### Core method steps
1. Load workbook sheets and validate required sheet mapping.
2. Extract KPI metadata (pillar/sub-pillar hierarchy, L1/L2 weights, scale descriptors).
3. Build country-KPI long table and compute weighted KPI contributions.
4. Aggregate to sub-pillar and composite scores.
5. Recompute rank ordering and compare with workbook references.
6. Cross-check metadata alignment with Canada gap CSV.

### Validation checks emphasized
- required sheet presence
- country count and KPI count consistency
- AHP L1/L2 weight checks and expected maps
- Canada target checks (sub-pillar/pillar/composite/rank)
- full rank-order consistency checks
- cross-file alignment checks with the Canada gap reference

### Implementation note
There is no baseline_master_table artifact in current implementation. Canonical baseline outputs are baseline_country_scores.csv and baseline_kpi_scores.csv.

In [ ]:
result = reproduce_baseline(
    workbook_path=WORKBOOK_PATH,
    canada_gap_csv_path=CANADA_GAP_PATH,
    output_dir=OUTPUT_DIR,
)

result.country_scores[[
    "rank",
    "country",
    "composite_score",
    "talent_score",
    "commercial_ecosystem_score",
    "government_strategy_score",
]].head(13)


In [ ]:
result.validation_checks[["section", "check", "passed"]]


In [ ]:
{name: str(path) for name, path in result.export_paths.items()}


## Reproducibility and Technical Notes

- Execute cells sequentially from top to bottom in a clean kernel.
- Keep input/output paths unchanged to preserve package-consistent exports.
- This notebook intentionally focuses on Phase 1 only; calibration, simulation, and attribution are handled in downstream notebooks.
- Validation outputs are part of the technical audit trail and should be retained with tables.

## Final Summary and Downstream Use

### What this notebook produced
- outputs/tables/baseline_country_scores.csv
- outputs/tables/baseline_kpi_scores.csv
- outputs/tables/validation_report.md

### How outputs are used in later phases
- Phase 3: baseline_kpi_scores.csv supports recommendation-to-KPI mapping integrity.
- Phase 4: baseline country/KPI tables provide simulation initial state.
- Phase 5: baseline KPI values provide benchmark-gap reference point.
- Phase 6: baseline outputs feed dashboard context and traceability artifacts.